In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [20]:
train = pd.read_csv(r"C:\Users\cyber\Downloads\spaceship-titanic\train.csv")
test = pd.read_csv(r"C:\Users\cyber\Downloads\spaceship-titanic\test.csv")

처음에 데이터가 어떤 형태인지 확인함

- 컬럼 내용 확인
- 결측치 확인
- 정답 컬럼 확인


In [21]:
print(train.head())
print(train.shape)
print(train.columns)
train.info()
print(train["Transported"].value_counts())
print(train.isnull().sum())

  PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4        303.0       70.0         151.0   565.0     2.0  Willy Santantines   

   Transported  
0        False  
1         True  
2        False  
3        False  
4         True  
(8

- 전체 데이터는 8693행 14열
- Transported 가 정답 컬럼
- 여러 컬럼에 결측치 존재함
- Transported 는 True, False 값이 거의 반반

## 베이스라인 모델 만들기

처음에는 너무 복잡하게 하지 않고 일단 돌아가는 분류 모델만 만듬

- PassengerId, Name, Cabin은 일단 제외
- 숫자형 결측치는 중앙값으로 채움
- 문자형 결측치는 최고 많이 나온 값으로 채움
- get_dummies()로 문자형을 숫자로 변경
- RandomForestClassifier로 학습

In [22]:
X = train.drop("Transported", axis=1)
y = train["Transported"]

def preprocess_baseline(df):
    df = df.copy()

    df = df.drop(["PassengerId", "Name", "Cabin"], axis=1)

    num_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())

    cat_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP"]
    for col in cat_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    df = pd.get_dummies(df, drop_first=True)

    return df

X_base = preprocess_baseline(train.drop("Transported", axis=1))
X_train_base, X_valid_base, y_train_base, y_valid_base = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train_base, y_train_base)

baseline_preds = baseline_model.predict(X_valid_base)
baseline_acc = accuracy_score(y_valid_base, baseline_preds)
print("Result : ", baseline_acc)

C:\Users\cyber\AppData\Local\Temp\ipykernel_36072\1545120015.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


Result :  0.7791834387579069


### 베이스라인 결과

베이스라인에서는 검증 정확도가 약 0.779 정도 나옴
목표 점수인 0.805를 맞추기 위해 튜닝 시작

1. PassengerId에서 그룹 정보 뽑기  
2. Cabin에서 Deck, Num, Side 나누기  
3. AgeBand 만들기  
4. CabinNumBand 만들기  
5. 소비 컬럼을 합쳐서 TotalSpend 만들기  
6. NoSpending 만들기  
7. 같은 그룹 기준으로 일부 결측치 보완하기

실제로 해보니 모든 아이디어가 다 도움이 되는 것은 아니었고 점수가 떨어지는 규칙은 다시 제거 함

실험을 하면서 점수가 조금씩 바뀌었음

- CabinNum, AgeBand 등을 추가한 버전 : 약 0.7976
- 그룹 기준 결측치 보완과 추가 feature를 넣은 버전 : 0.8056
- 그런데 TotalSpend == 0 이면 CryoSleep = True로 채우는 규칙은 오히려 성능을 떨어뜨림
- 그 규칙을 제거하고 CabinNumBand까지 사용한 최종 버전에서 0.809가 나옴

feature를 무조건 많이 넣는 것이 아니라 실제로 도움이 되는 것만 남기것이 중요

## 최종 코드

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

train = pd.read_csv(r"C:\Users\cyber\Downloads\spaceship-titanic\train.csv")
test = pd.read_csv(r"C:\Users\cyber\Downloads\spaceship-titanic\test.csv")

y = train["Transported"]

def preprocess(df):
    df = df.copy()

    # 그룹 번호
    df["Group"] = df["PassengerId"].str.split("_").str[0]

    # 그룹 크기
    group_counts = df["Group"].value_counts()
    df["GroupSize"] = df["Group"].map(group_counts)

    # Cabin 분리
    df["CabinDeck"] = df["Cabin"].str.split("/").str[0]
    df["CabinNum"] = pd.to_numeric(df["Cabin"].str.split("/").str[1], errors="coerce")
    df["CabinSide"] = df["Cabin"].str.split("/").str[2]

    # 그룹 기준 같은 값으로 결측치 채움
    group_cat_cols = ["HomePlanet", "Destination", "CabinDeck", "CabinSide"]
    for col in group_cat_cols:
        df[col] = df.groupby("Group")[col].transform(
            lambda x: x.fillna(x.mode()[0]) if not x.mode().empty else x
        )

    # CryoSleep=True 이면 소비 결측치는 0으로 채우기
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    for col in spend_cols:
        df.loc[df["CryoSleep"] == True, col] = df.loc[df["CryoSleep"] == True, col].fillna(0)

    # 숫자형 결측치를 중앙값으로 채우기
    num_cols = ["Age", "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck", "CabinNum"]
    for col in num_cols:
        df[col] = df[col].fillna(df[col].median())

    # Age를 연령대 구간으로 나눠 모델이 나이대별 패턴을 쉽게 학습하도록 함
    # 여러 구간을 테스트 해봤지만 너무 많아도 너무 적어도 결과 값 이 떨어졌음
    # 0-12: 어린이, 13-18: 청소년, 19-30: 청년, 31-50: 중년, 51-80: 노년 기준으로 분류했음
    # 10,20,30,40,50,60,70,80 으로 나눠봤지만 오히려 성능을 떨어 뜨림
    df["AgeBand"] = pd.cut(df["Age"], bins=[0, 12, 18, 30, 50, 80], labels=False)
    
    # CabinNum을 6개 구간으로 나눠 객실 위치대 패턴을 반영하도록 함
    df["CabinNumBand"] = pd.cut(df["CabinNum"], bins=6, labels=False)

    df["TotalSpend"] = (
        df["RoomService"]
        + df["FoodCourt"]
        + df["ShoppingMall"]
        + df["Spa"]
        + df["VRDeck"]
    )

    df["NoSpending"] = (df["TotalSpend"] == 0)

    # 소비가 0이면 CryoSleep 결측치를 True로 채우기
    #df.loc[df["CryoSleep"].isna() & (df["TotalSpend"] == 0), "CryoSleep"] = True

    # 남은 문자형 결측치 채우기
    cat_cols = ["HomePlanet", "CryoSleep", "Destination", "VIP", "CabinDeck", "CabinSide"]
    for col in cat_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

    # 사용하지 않을 컬럼 제거
    df = df.drop(["PassengerId", "Name", "Cabin", "Group"], axis=1)

    # 문자형 -> 숫자형 변환
    df = pd.get_dummies(df, drop_first=True)

    return df

# 4. 전처리
X = preprocess(train.drop("Transported", axis=1))
test_processed = preprocess(test)

# 5. train,test 컬럼 맞추기
test_processed = test_processed.reindex(columns=X.columns, fill_value=0)

# 6. 학습용,검증용 분리
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=700,
    max_depth=14,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

model.fit(X_train, y_train)

preds = model.predict(X_valid)
acc = accuracy_score(y_valid, preds)
print("정확도:", acc)

final_model = RandomForestClassifier(
    n_estimators=700,
    max_depth=14,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

final_model.fit(X, y)

test_preds = final_model.predict(test_processed)
sample_sub = pd.read_csv(r"C:\Users\cyber\Downloads\spaceship-titanic\sample_submission.csv")
sample_sub["Transported"] = pd.Series(test_preds).astype(str)
sample_sub.to_csv("submission.csv", index=False)
print(sample_sub.head())


C:\Users\cyber\AppData\Local\Temp\ipykernel_36072\1081087438.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])
C:\Users\cyber\AppData\Local\Temp\ipykernel_36072\1081087438.py:68: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(df[col].mode()[0])


정확도: 0.8096607245543416
  PassengerId Transported
0     0013_01        True
1     0018_01       False
2     0019_01        True
3     0021_01        True
4     0023_01        True


## 최종 

최종적으로 검증 정확도는 약 0.809까지 올릴 수 있었음

- Cabin을 그냥 버리기보다 나누어서 쓰는 것이 좋았음
- PassengerId에서 그룹 정보를 뽑는 것도 도움이 되었음
- AgeBand, CabinNumBand, TotalSpend, NoSpending 같은 파생변수도 도움이 되었음
- 반대로 TotalSpend == 0 이면 CryoSleep = True 라고 강제로 채우는 규칙은 오히려 성능을 떨어뜨림